
# Roxy notebook example: Local repetition and redundancy descriptors

This notebook is a **reference implementation example** for the **local repetition and redundancy descriptor family** in Roxy.

These descriptors try to quantify whether a sequence contains **recurrent local patterns**, **redundant subsequences**, or **short repeated motifs** that appear multiple times along the chain.

They are closely related to complexity descriptors, but here the emphasis is more specific:

- repeated local words
- recurrence of short subsequences
- local redundancy burden
- repeated neighborhood patterns
- recurrence concentration

## Covered outputs

This notebook implements examples such as:

- repeated k-mer fraction
- unique k-mer ratio
- redundancy score
- top repeated k-mer count
- top repeated k-mer frequency
- repeated-window burden
- duplicate window fraction
- motif recurrence concentration
- local recurrence entropy
- repeated pattern span
- repeated block density
- class-style implementation for later migration into Roxy

The notebook is written as a **clean teaching implementation** so it can later become part of the real Roxy package.


In [1]:

from collections import Counter, defaultdict
from itertools import groupby

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "rep_1",
            "rep_2",
            "rep_3",
            "rep_4",
            "rep_5",
            "rep_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,rep_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,rep_2,GGGGGGGGGGGGGGG,B
2,rep_3,KRRKRRKRRKRRDDDDEE,A
3,rep_4,ACDEFGHIKLMNPQRSTVWY,B
4,rep_5,PPPPGSSSSSTTTTNNQQQ,A
5,rep_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def kmers(seq: str, k: int):
    if len(seq) < k:
        return []
    return [seq[i:i+k] for i in range(len(seq) - k + 1)]


def repeated_fraction(words) -> float:
    if len(words) == 0:
        return np.nan
    counts = Counter(words)
    repeated = sum(v for v in counts.values() if v > 1)
    return repeated / len(words)


def unique_fraction(words) -> float:
    if len(words) == 0:
        return np.nan
    return len(set(words)) / len(words)


def redundancy_score(words) -> float:
    if len(words) == 0:
        return np.nan
    return 1.0 - unique_fraction(words)


def top_repeated_count(words) -> float:
    if len(words) == 0:
        return np.nan
    counts = Counter(words)
    return float(max(counts.values()))


def top_repeated_frequency(words) -> float:
    if len(words) == 0:
        return np.nan
    return top_repeated_count(words) / len(words)


def duplicate_window_fraction(seq: str, window: int = 5) -> float:
    words = kmers(seq, window)
    if len(words) == 0:
        return np.nan
    counts = Counter(words)
    duplicate_windows = sum(1 for w in words if counts[w] > 1)
    return duplicate_windows / len(words)


def repeated_word_burden(seq: str, k: int = 2) -> float:
    words = kmers(seq, k)
    if len(words) == 0:
        return np.nan
    counts = Counter(words)
    burden = sum(count for _, count in counts.items() if count > 1)
    return burden / len(words)


def recurrence_entropy(words) -> float:
    if len(words) == 0:
        return np.nan
    counts = Counter(words)
    repeated_counts = np.array([v for v in counts.values() if v > 1], dtype=float)
    if len(repeated_counts) == 0:
        return 0.0
    probs = repeated_counts / repeated_counts.sum()
    return float(-(probs * np.log2(probs)).sum())


def repeated_word_span(seq: str, k: int = 2) -> float:
    words = kmers(seq, k)
    if len(words) == 0:
        return np.nan

    positions = defaultdict(list)
    for i, w in enumerate(words):
        positions[w].append(i)

    spans = []
    for _, pos in positions.items():
        if len(pos) > 1:
            spans.append(pos[-1] - pos[0])

    if len(spans) == 0:
        return np.nan
    return float(np.mean(spans))


def repeated_word_span_norm(seq: str, k: int = 2) -> float:
    span = repeated_word_span(seq, k=k)
    if len(seq) == 0 or np.isnan(span):
        return np.nan
    return span / len(seq)


def repeated_block_density(seq: str, k: int = 2) -> float:
    words = kmers(seq, k)
    if len(words) == 0:
        return np.nan
    counts = Counter(words)
    binary = [1 if counts[w] > 1 else 0 for w in words]
    if len(binary) == 0:
        return np.nan
    return sum(binary) / len(binary)


def longest_redundant_block(seq: str, k: int = 2) -> float:
    words = kmers(seq, k)
    if len(words) == 0:
        return np.nan
    counts = Counter(words)
    binary = [1 if counts[w] > 1 else 0 for w in words]
    runs = [len(list(group)) for value, group in groupby(binary) if value == 1]
    if len(runs) == 0:
        return 0.0
    return float(max(runs))


def recurrence_concentration(words) -> float:
    if len(words) == 0:
        return np.nan
    counts = Counter(words)
    repeated = [v for v in counts.values() if v > 1]
    if len(repeated) == 0:
        return 0.0
    total_repeated = sum(repeated)
    return max(repeated) / total_repeated


def local_redundancy_window_profile(seq: str, outer_window: int = 8, inner_k: int = 2) -> list:
    if len(seq) < outer_window or outer_window < inner_k:
        return []
    profiles = []
    for i in range(len(seq) - outer_window + 1):
        sub = seq[i:i+outer_window]
        words = kmers(sub, inner_k)
        profiles.append(redundancy_score(words))
    return profiles


def mean_local_redundancy(seq: str, outer_window: int = 8, inner_k: int = 2) -> float:
    profile = local_redundancy_window_profile(seq, outer_window=outer_window, inner_k=inner_k)
    if len(profile) == 0:
        return np.nan
    return float(np.mean(profile))


def std_local_redundancy(seq: str, outer_window: int = 8, inner_k: int = 2) -> float:
    profile = local_redundancy_window_profile(seq, outer_window=outer_window, inner_k=inner_k)
    if len(profile) == 0:
        return np.nan
    return float(np.std(profile, ddof=0))


## Core descriptor function

In [5]:

def local_repetition_descriptors(seq: str) -> dict:
    seq = clean_sequence(seq)

    words2 = kmers(seq, 2)
    words3 = kmers(seq, 3)
    words4 = kmers(seq, 4)

    out = {
        "rep_length": len(seq),
        "rep_valid_residue_count": len(seq),

        "rep_repeated_fraction_k2": repeated_fraction(words2),
        "rep_repeated_fraction_k3": repeated_fraction(words3),
        "rep_repeated_fraction_k4": repeated_fraction(words4),

        "rep_unique_fraction_k2": unique_fraction(words2),
        "rep_unique_fraction_k3": unique_fraction(words3),
        "rep_unique_fraction_k4": unique_fraction(words4),

        "rep_redundancy_score_k2": redundancy_score(words2),
        "rep_redundancy_score_k3": redundancy_score(words3),
        "rep_redundancy_score_k4": redundancy_score(words4),

        "rep_top_count_k2": top_repeated_count(words2),
        "rep_top_count_k3": top_repeated_count(words3),
        "rep_top_freq_k2": top_repeated_frequency(words2),
        "rep_top_freq_k3": top_repeated_frequency(words3),

        "rep_duplicate_window_fraction_w5": duplicate_window_fraction(seq, window=5),
        "rep_duplicate_window_fraction_w6": duplicate_window_fraction(seq, window=6),

        "rep_repeated_word_burden_k2": repeated_word_burden(seq, k=2),
        "rep_repeated_word_burden_k3": repeated_word_burden(seq, k=3),

        "rep_recurrence_entropy_k2": recurrence_entropy(words2),
        "rep_recurrence_entropy_k3": recurrence_entropy(words3),

        "rep_repeated_span_k2": repeated_word_span(seq, k=2),
        "rep_repeated_span_k3": repeated_word_span(seq, k=3),
        "rep_repeated_span_norm_k2": repeated_word_span_norm(seq, k=2),
        "rep_repeated_span_norm_k3": repeated_word_span_norm(seq, k=3),

        "rep_repeated_block_density_k2": repeated_block_density(seq, k=2),
        "rep_repeated_block_density_k3": repeated_block_density(seq, k=3),

        "rep_longest_redundant_block_k2": longest_redundant_block(seq, k=2),
        "rep_longest_redundant_block_k3": longest_redundant_block(seq, k=3),

        "rep_recurrence_concentration_k2": recurrence_concentration(words2),
        "rep_recurrence_concentration_k3": recurrence_concentration(words3),

        "rep_local_redundancy_mean_w8_k2": mean_local_redundancy(seq, outer_window=8, inner_k=2),
        "rep_local_redundancy_std_w8_k2": std_local_redundancy(seq, outer_window=8, inner_k=2),
        "rep_local_redundancy_mean_w10_k2": mean_local_redundancy(seq, outer_window=10, inner_k=2),
        "rep_local_redundancy_std_w10_k2": std_local_redundancy(seq, outer_window=10, inner_k=2),
    }

    return out


## Functional usage on one sequence

In [6]:

example = local_repetition_descriptors(df_demo.loc[0, "sequence"])
list(example.items())[:18]


[('rep_length', 24),
 ('rep_valid_residue_count', 24),
 ('rep_repeated_fraction_k2', 0.08695652173913043),
 ('rep_repeated_fraction_k3', 0.0),
 ('rep_repeated_fraction_k4', 0.0),
 ('rep_unique_fraction_k2', 0.9565217391304348),
 ('rep_unique_fraction_k3', 1.0),
 ('rep_unique_fraction_k4', 1.0),
 ('rep_redundancy_score_k2', 0.04347826086956519),
 ('rep_redundancy_score_k3', 0.0),
 ('rep_redundancy_score_k4', 0.0),
 ('rep_top_count_k2', 2.0),
 ('rep_top_count_k3', 1.0),
 ('rep_top_freq_k2', 0.08695652173913043),
 ('rep_top_freq_k3', 0.045454545454545456),
 ('rep_duplicate_window_fraction_w5', 0.0),
 ('rep_duplicate_window_fraction_w6', 0.0),
 ('rep_repeated_word_burden_k2', 0.08695652173913043)]

## Apply repetition descriptors to the full dataset

In [7]:

df_rep = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(local_repetition_descriptors).apply(pd.Series),
    ],
    axis=1,
)

df_rep.head()


,sequence_id,sequence,label,rep_length,rep_valid_residue_count,rep_repeated_fraction_k2,rep_repeated_fraction_k3,rep_repeated_fraction_k4,rep_unique_fraction_k2,rep_unique_fraction_k3,...,rep_repeated_block_density_k2,rep_repeated_block_density_k3,rep_longest_redundant_block_k2,rep_longest_redundant_block_k3,rep_recurrence_concentration_k2,rep_recurrence_concentration_k3,rep_local_redundancy_mean_w8_k2,rep_local_redundancy_std_w8_k2,rep_local_redundancy_mean_w10_k2,rep_local_redundancy_std_w10_k2
0,rep_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,0.086957,0.000000,0.000,0.956522,1.000000,...,0.086957,0.000000,1.0,0.0,1.000000,0.000000,0.042017,0.065092,0.051852,5.543196e-02
1,rep_2,GGGGGGGGGGGGGGG,B,15.0,15.0,1.000000,1.000000,1.000,0.071429,0.076923,...,1.000000,1.000000,14.0,13.0,1.000000,1.000000,0.857143,0.000000,0.888889,1.110223e-16
2,rep_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,0.823529,0.750000,0.600,0.411765,0.500000,...,0.823529,0.750000,11.0,10.0,0.285714,0.333333,0.428571,0.136209,0.493827,1.491734e-01
3,rep_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,0.000000,0.000000,0.000,1.000000,1.000000,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00
4,rep_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,0.666667,0.411765,0.125,0.555556,0.764706,...,0.666667,0.411765,4.0,3.0,0.333333,0.428571,0.404762,0.098169,0.422222,1.088662e-01


## Inspect repetition descriptor columns

In [8]:

rep_cols = [c for c in df_rep.columns if c.startswith("rep_") and c not in {"rep_length", "rep_valid_residue_count"}]
len(rep_cols), rep_cols[:16]


(33,
 ['rep_repeated_fraction_k2',
  'rep_repeated_fraction_k3',
  'rep_repeated_fraction_k4',
  'rep_unique_fraction_k2',
  'rep_unique_fraction_k3',
  'rep_unique_fraction_k4',
  'rep_redundancy_score_k2',
  'rep_redundancy_score_k3',
  'rep_redundancy_score_k4',
  'rep_top_count_k2',
  'rep_top_count_k3',
  'rep_top_freq_k2',
  'rep_top_freq_k3',
  'rep_duplicate_window_fraction_w5',
  'rep_duplicate_window_fraction_w6',
  'rep_repeated_word_burden_k2'])

In [9]:

df_rep[
    [
        "sequence_id",
        "rep_repeated_fraction_k2",
        "rep_redundancy_score_k2",
        "rep_top_freq_k2",
        "rep_duplicate_window_fraction_w5",
        "rep_recurrence_entropy_k2",
        "rep_repeated_span_norm_k2",
        "rep_local_redundancy_mean_w8_k2",
    ]
]


,sequence_id,rep_repeated_fraction_k2,rep_redundancy_score_k2,rep_top_freq_k2,rep_duplicate_window_fraction_w5,rep_recurrence_entropy_k2,rep_repeated_span_norm_k2,rep_local_redundancy_mean_w8_k2
0,rep_1,0.086957,0.043478,0.086957,0.000000,-0.000000,0.083333,0.042017
1,rep_2,1.000000,0.928571,1.000000,1.000000,-0.000000,0.866667,0.857143
2,rep_3,0.823529,0.588235,0.235294,0.571429,1.985228,0.361111,0.428571
3,rep_4,0.000000,0.000000,0.052632,0.000000,0.000000,NaN,0.000000
4,rep_5,0.666667,0.444444,0.222222,0.000000,1.959148,0.105263,0.404762
5,rep_6,0.000000,0.000000,0.050000,0.000000,0.000000,NaN,0.000000


## Dataset-level summary

In [10]:

rep_summary = (
    df_rep[rep_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

rep_summary.head(15)


,descriptor,mean_value
0,rep_repeated_span_k3,6.277778
1,rep_repeated_span_k2,5.875000
2,rep_longest_redundant_block_k2,5.000000
3,rep_longest_redundant_block_k3,4.333333
4,rep_top_count_k2,4.333333
5,rep_top_count_k3,3.833333
6,rep_unique_fraction_k4,0.770139
7,rep_unique_fraction_k3,0.723605
8,rep_unique_fraction_k2,0.665878
9,rep_recurrence_entropy_k2,0.657396


## Sanity checks

In [11]:

assert "rep_repeated_fraction_k2" in df_rep.columns
assert "rep_redundancy_score_k3" in df_rep.columns
assert "rep_top_freq_k2" in df_rep.columns
assert "rep_duplicate_window_fraction_w5" in df_rep.columns
assert "rep_recurrence_entropy_k2" in df_rep.columns
assert "rep_local_redundancy_mean_w8_k2" in df_rep.columns
assert df_rep["rep_length"].min() > 0

print(f"Number of repetition/redundancy descriptor columns: {len(rep_cols)}")
print("Repetition and local redundancy descriptor checks passed.")


Number of repetition/redundancy descriptor columns: 33
Repetition and local redundancy descriptor checks passed.


## Class-style implementation closer to the real package

In [12]:

class LocalRepetitionDescriptors:
    """Example class-style repetition/redundancy implementation for later migration into Roxy."""

    def transform_sequence(self, seq: str) -> dict:
        return local_repetition_descriptors(seq)

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


rep_transformer = LocalRepetitionDescriptors()
rep_matrix = rep_transformer.transform(df_demo["sequence"].tolist())
rep_matrix.head()


,rep_length,rep_valid_residue_count,rep_repeated_fraction_k2,rep_repeated_fraction_k3,rep_repeated_fraction_k4,rep_unique_fraction_k2,rep_unique_fraction_k3,rep_unique_fraction_k4,rep_redundancy_score_k2,rep_redundancy_score_k3,...,rep_repeated_block_density_k2,rep_repeated_block_density_k3,rep_longest_redundant_block_k2,rep_longest_redundant_block_k3,rep_recurrence_concentration_k2,rep_recurrence_concentration_k3,rep_local_redundancy_mean_w8_k2,rep_local_redundancy_std_w8_k2,rep_local_redundancy_mean_w10_k2,rep_local_redundancy_std_w10_k2
0,24,24,0.086957,0.000000,0.000,0.956522,1.000000,1.000000,0.043478,0.000000,...,0.086957,0.000000,1.0,0.0,1.000000,0.000000,0.042017,0.065092,0.051852,5.543196e-02
1,15,15,1.000000,1.000000,1.000,0.071429,0.076923,0.083333,0.928571,0.923077,...,1.000000,1.000000,14.0,13.0,1.000000,1.000000,0.857143,0.000000,0.888889,1.110223e-16
2,18,18,0.823529,0.750000,0.600,0.411765,0.500000,0.600000,0.588235,0.500000,...,0.823529,0.750000,11.0,10.0,0.285714,0.333333,0.428571,0.136209,0.493827,1.491734e-01
3,20,20,0.000000,0.000000,0.000,1.000000,1.000000,1.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00
4,19,19,0.666667,0.411765,0.125,0.555556,0.764706,0.937500,0.444444,0.235294,...,0.666667,0.411765,4.0,3.0,0.333333,0.428571,0.404762,0.098169,0.422222,1.088662e-01


## Merge transformer output back to the dataset

In [13]:

df_rep_class = pd.concat([df_demo, rep_matrix], axis=1)
df_rep_class.head()


,sequence_id,sequence,label,rep_length,rep_valid_residue_count,rep_repeated_fraction_k2,rep_repeated_fraction_k3,rep_repeated_fraction_k4,rep_unique_fraction_k2,rep_unique_fraction_k3,...,rep_repeated_block_density_k2,rep_repeated_block_density_k3,rep_longest_redundant_block_k2,rep_longest_redundant_block_k3,rep_recurrence_concentration_k2,rep_recurrence_concentration_k3,rep_local_redundancy_mean_w8_k2,rep_local_redundancy_std_w8_k2,rep_local_redundancy_mean_w10_k2,rep_local_redundancy_std_w10_k2
0,rep_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,0.086957,0.000000,0.000,0.956522,1.000000,...,0.086957,0.000000,1.0,0.0,1.000000,0.000000,0.042017,0.065092,0.051852,5.543196e-02
1,rep_2,GGGGGGGGGGGGGGG,B,15,15,1.000000,1.000000,1.000,0.071429,0.076923,...,1.000000,1.000000,14.0,13.0,1.000000,1.000000,0.857143,0.000000,0.888889,1.110223e-16
2,rep_3,KRRKRRKRRKRRDDDDEE,A,18,18,0.823529,0.750000,0.600,0.411765,0.500000,...,0.823529,0.750000,11.0,10.0,0.285714,0.333333,0.428571,0.136209,0.493827,1.491734e-01
3,rep_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,0.000000,0.000000,0.000,1.000000,1.000000,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00
4,rep_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,0.666667,0.411765,0.125,0.555556,0.764706,...,0.666667,0.411765,4.0,3.0,0.333333,0.428571,0.404762,0.098169,0.422222,1.088662e-01



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move helper logic into `roxy/sequence/complexity.py` or a dedicated `redundancy.py`
- expose a class such as `LocalRepetitionDescriptors`
- allow configurable:
  - k values to track
  - outer windows for local redundancy
  - which repetition summaries to compute
- add tests for:
  - empty sequences
  - highly repetitive sequences
  - fully unique sequences
  - strongly periodic sequences
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [ ]:
# df_rep.to_csv("demo_local_repetition_descriptors.csv", index=False)
